# 用模型辅助检查回答

LLM-as-Judge 是让另一个模型根据明确规则检查回答。它适合处理“是否回应问题”、“是否有资料支持”这类很难用一个关键词判断的项目，但模型评分也会错。

本页分开呈现三部分：前面的 Cohen’s Kappa 数组是手写标签，只演示一致性计算；中间保留上一轮两条样本的失败记录与复查；末尾扩展为 10 条小型校准集，使用根目录 `.env` 中固定的 `glm-4-flash` 做真实 LLM-as-Judge 调用。扩展集覆盖正确、部分正确、无依据、引用错配和应拒答；每条期望是仓库内的教学预设标签，记录为 `human_verified=false`，不是独立人工语义核验、外部人工研究或独立金标准。

## 先固定评分规则

一个可复查的规则至少包含：输入有哪些，每个分数表示什么，应该返回哪些字段，以及资料不足时怎样处理。可以要求模型先列出支持或反对的原文，再给分；但最终还要比较它和人工判断的一致程度。

In [1]:
human = [1, 1, 0, 1, 0, 0, 1, 0]
judge = [1, 1, 1, 1, 0, 0, 1, 0]

def cohen_kappa(left, right):
    observed = sum(a == b for a, b in zip(left, right)) / len(left)
    left_pos = sum(left) / len(left)
    right_pos = sum(right) / len(right)
    expected = left_pos * right_pos + (1-left_pos) * (1-right_pos)
    return (observed - expected) / (1 - expected)

print("一致数量：", sum(a == b for a, b in zip(human, judge)), "/", len(human))
print("Cohen's kappa：", round(cohen_kappa(human, judge), 3))
print("不一致的样本编号：", [index for index, (a, b) in enumerate(zip(human, judge), start=1) if a != b])

一致数量： 7 / 8
Cohen's kappa： 0.75
不一致的样本编号： [3]


先用人工检查的小批问题校准，再扩大使用。评分模型、Prompt 或分数定义变更后，一致性需要重新检查。本页的 8 个标签只用于演示计算，不是对某个真实评分模型的结论。



## LLM-as-Judge 的评分设计、直接评分和逐步评分

LLM-as-Judge（模型评审）适合检查“是否切题、是否有资料支持、是否覆盖多个要求”等难以用字符串规则判断的项目。它不是事实来源，也不是天然可靠的标注员。Prompt 至少要固定角色、输入字段、评分锚点、资料不足的处理、输出格式和拒答规则。

直接评分只要求模型返回分数，成本较低但难以复查；逐步评分可以先列出每项结论及其原文依据，再给分，更容易发现漏项，但输出更长、费用更高。比较两种 Prompt 时要使用同一批问题，再看平均分、分数波动和人工评分是否一致。一次逐步评分更高，不能说明它对所有问题都更准。还要防止模型偏爱排在前面的答案、较长的答案、自己生成的答案或特定格式；可以交换答案顺序、限制长度并抽样人工复核。

本页保留 8 个手写二分类标签演示 Cohen’s Kappa；它们不是外部评审模型的实测结果。真实模型调用先保留下一段的两条历史校准样本，页面末尾再用 10 条新样本覆盖更多错误类型。


## LLM-as-Judge 的代码写法

下面先展示评审模型的输入、输出和格式检查代码写法；中间单元固定同一问题、资料和两种回答，保留历史真实返回；末尾单元再对 10 条扩展样本真实调用并保存原始返回，与教程作者期望逐条复核。

```python
import json

DIRECT_JUDGE_PROMPT = """你是评审员，不是回答者。根据问题、资料和回答评分。
0=没有回应问题或主要内容错误；1=部分回应或部分有资料支持；2=完整回应且主要结论都有资料依据。
只输出 JSON：{{\"score\": 0|1|2, \"reason\": \"一句话\", \"evidence\": [\"原文短语\"]}}。
问题：{question}\n资料：{context}\n回答：{answer}"""

STEPWISE_JUDGE_PROMPT = """先把回答拆成不超过 5 条可核对的结论；为每条结论列出资料中的原文短语，找不到就写 NOTHING_FOUND；
按有据且切题的结论比例给 0/1/2 分。只输出 JSON，字段为 claims、score、reason。
问题：{question}\n资料：{context}\n回答：{answer}"""

def parse_judge_json(raw):
    text = str(raw).strip()
    fence = chr(96) * 3
    if text.startswith(fence + "json") and text.endswith(fence):
        text = text[len(fence + "json"):-len(fence)].strip()
    obj = json.loads(text)
    if not isinstance(obj, dict):
        raise ValueError("judge 输出必须是 JSON 对象")
    score = obj.get("score")
    if isinstance(score, bool) or score not in (0, 1, 2):
        raise ValueError("score 必须是 0、1 或 2")
    reason = obj.get("reason")
    if not isinstance(reason, str) or not reason.strip():
        raise ValueError("reason 必须是非空字符串")
    claims = obj.get("claims", [])
    evidence = obj.get("evidence", [])
    if not isinstance(claims, list):
        raise ValueError("claims 必须是列表")
    if not isinstance(evidence, list):
        raise ValueError("evidence 必须是列表")
    if any(not isinstance(item, str) for item in evidence):
        raise ValueError("evidence 每项必须是字符串")
    if any(not item.strip() for item in evidence):
        raise ValueError("evidence 每项必须是非空字符串")
    for claim in claims:
        if not isinstance(claim, dict) or not isinstance(claim.get("evidence", []), list):
            raise ValueError("claims 中的 evidence 必须是列表")
    return {"score": score, "reason": reason, "evidence": evidence, "claims": claims}

def judge_one(question, context, answer, llm_call, stepwise=True):
    template = STEPWISE_JUDGE_PROMPT if stepwise else DIRECT_JUDGE_PROMPT
    return parse_judge_json(llm_call(template.format(question=question, context=context, answer=answer)))

import json

def cohen_kappa_multiclass(left, right):
    """不依赖 sklearn 的 Cohen's Kappa；输入为等长离散标签。"""
    if len(left) != len(right) or not left: raise ValueError("两组标签必须等长且非空")
    observed = sum(a == b for a, b in zip(left, right)) / len(left)
    labels = set(left) | set(right)
    expected = sum((left.count(label) / len(left)) * (right.count(label) / len(right)) for label in labels)
    return (observed - expected) / (1 - expected) if expected < 1 else 1.0

# 先用人工标注的小批样本校准，再扩大到整套评估集；换模型、Prompt 或标签定义后要重校准。
```


## 评分规则、两种问法和已知局限

先把评分协议写下来，再调用评审模型：输入固定为问题（`question`）、资料（`context`）、回答（`answer`），必要时另列参考答案（`ground_truth`）；0 分表示没有回答或主要结论错误，1 分表示部分回答或只有部分结论有依据，2 分表示完整回答且主要结论都有依据。输出必须包含分数、简短理由和原文证据；资料没有答案时，应该检查模型是否明确拒答，不能因为回答写得流畅就给高分。

| 做法 | 提示词要求 | 优点 | 风险 |
| --- | --- | --- | --- |
| 直接评分 | 读完输入后直接返回分数 | 省 token、速度快 | 理由少，漏掉某个结论后不易追查 |
| 逐步评分 | 先列可核对的结论，再为每条结论找原文证据，再汇总分数 | 更容易发现漏答和无依据结论 | 输出长、费用高，模型的中间分析仍可能出错 |

两种做法必须在同一批问题、相同资料、相同分数定义上比较；一次实验看到逐步评分更高，不能推出它在所有模型和任务上都更好。还要防止位置偏差、冗长偏差、自我偏好和格式偏差：交换候选答案顺序、限制长度、抽样人工复核，并把评审模型当作有误差的测量工具。

In [2]:
import json

def cohen_kappa_multiclass(left, right):
    """只用标准库计算离散标签的一致性；两组标签必须来自同一批样本。"""
    if len(left) != len(right) or not left:
        raise ValueError("两组标签必须等长且非空")
    observed = sum(a == b for a, b in zip(left, right)) / len(left)
    labels = set(left) | set(right)
    expected = sum((left.count(label) / len(left)) * (right.count(label) / len(right))
                    for label in labels)
    return (observed - expected) / (1 - expected) if expected < 1 else 1.0

def summarize_judge_modes(direct_scores, stepwise_scores, human_scores=None):
    """汇总两种评分的均值、差异和（可选）与人工标签的 Kappa。"""
    if len(direct_scores) != len(stepwise_scores):
        raise ValueError("直接评分和逐步评分必须使用同一批样本")
    result = {
        "n": len(direct_scores),
        "direct_mean": sum(direct_scores) / len(direct_scores) if direct_scores else 0.0,
        "stepwise_mean": sum(stepwise_scores) / len(stepwise_scores) if stepwise_scores else 0.0,
        "stepwise_minus_direct": (sum(stepwise_scores) - sum(direct_scores)) / len(direct_scores)
            if direct_scores else 0.0,
    }
    if human_scores is not None:
        if len(human_scores) != len(direct_scores):
            raise ValueError("人工标签必须与同一批样本对齐")
        result["direct_kappa"] = cohen_kappa_multiclass(human_scores, direct_scores)
        result["stepwise_kappa"] = cohen_kappa_multiclass(human_scores, stepwise_scores)
    return result

direct_demo = [2, 1, 0, 2]
stepwise_demo = [2, 2, 0, 2]
human_demo = [2, 1, 0, 2]
summary = summarize_judge_modes(direct_demo, stepwise_demo, human_demo)
print(
    f"评分对照：{summary['n']} 个问题；直接评分平均 {summary['direct_mean']:.2f}，"
    f"逐步评分平均 {summary['stepwise_mean']:.2f}，逐步评分高 {summary['stepwise_minus_direct']:.2f}；"
    f"与人工标签的一致性分别为 {summary['direct_kappa']:.2f} 和 {summary['stepwise_kappa']:.2f}。"
)
# 上面的标签是本地演示数据，不调用评审模型，也不替换前面已保存的 8 个标签输出。

PROBE_DIRECT_PROMPT = "只输出 JSON：{{\"score\": 0|1|2, \"reason\": \"一句话\", \"evidence\": [\"原文短语\"]}}。问题：{question}\n资料：{context}\n回答：{answer}"
def judge_one(question, context, answer, llm_call, stepwise=True):
    # 双花括号让 format 保留 JSON 对象；此探针只验证模板，不计入真实校准。
    prompt = PROBE_DIRECT_PROMPT.format(question=question, context=context, answer=answer)
    return json.loads(llm_call(prompt))

probe_prompt = PROBE_DIRECT_PROMPT.format(question='一个有依据的问题', context='资料片段', answer='依据资料作答')
assert all(label in probe_prompt for label in ('问题：', '资料：', '回答：'))
print('格式探针：直接评分模板可完成字段插值；不调用评审模型，也不计入下面的真实校准。')


评分对照：4 个问题；直接评分平均 1.25，逐步评分平均 1.50，逐步评分高 0.25；与人工标签的一致性分别为 1.00 和 0.56。
格式探针：直接评分模板可完成字段插值；不调用评审模型，也不计入下面的真实校准。


## 第一次真实校准的失败记录（保留）

上一轮使用同一问题、同一页资料和两条回答完成了 4 次 `glm-4-flash` 调用。下面保留它的失败过程，后面的改进复查不会把它改写成成功：有原文依据的回答，直接评分为 2、逐项评分为 2，均与人工预期一致；明确错误的回答，直接评分为 0，但逐项评分错误地给了 2，且把被资料语义反驳的结论列为有据。原逐项评分只有 1/2 条与人工预期一致。

这说明“出现相关词”不等于“支持该结论”：原逐项结果引用了包含相关词的句子，却没有判断回答的语义方向。改进后的复查只增加 2 次逐项评分调用，并单独做引用核验。

In [3]:
import json
import sys
from pathlib import Path

course_root = Path.cwd()
for folder in (course_root, *course_root.parents):
    if (folder / 'data' / 'dataset/manifest.json').is_file():
        course_root = folder
        break
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))

from common.eval_utils import load_query_catalog, load_pdf_pages
from common.nontraining_utils import llm_call, load_annotation

DIRECT_CALIBRATION_PROMPT = """你是严格的资料依据评审员，不是回答者。
0=没有回应问题或主要结论错误；1=部分回应或部分有资料支持；2=完整回应且主要结论都有资料依据。
只输出 JSON：{{\"score\": 0|1|2, \"reason\": \"一句话\", \"evidence\": [\"资料中的短语\"]}}。
问题：{question}\n资料：{context}\n回答：{answer}"""

STEPWISE_CALIBRATION_PROMPT = """你是严格的资料依据评审员。先列出回答中不超过 5 条可核对的结论，再逐条从资料找证据；找不到就写 NOTHING_FOUND。最后按有据且切题的结论比例给 0/1/2 分；即使有三条结论，也只能给最高分 2，绝对不要输出 3。score 只能是整数 0、1 或 2。
只输出 JSON：{{\"claims\": [{{\"claim\": \"结论\", \"evidence\": [\"短语或 NOTHING_FOUND\"], \"supported\": true|false}}], \"score\": 0|1|2, \"reason\": \"一句话\"}}。
问题：{question}\n资料：{context}\n回答：{answer}"""

def parse_real_judge(raw):
    text = str(raw).strip()
    if text.startswith('```json') and text.endswith('```'):
        text = text[len('```json'):-3].strip()
    value = json.loads(text)
    if not isinstance(value, dict):
        raise ValueError('模型返回的不是 JSON 对象')
    score = value.get('score')
    if isinstance(score, bool) or score not in (0, 1, 2):
        raise ValueError('模型返回的分数不是 0、1、2')
    reason = value.get('reason')
    if not isinstance(reason, str) or not reason.strip():
        raise ValueError('模型返回的 reason 必须是非空字符串')
    if 'evidence' in value and not isinstance(value['evidence'], list):
        raise ValueError('模型返回的 evidence 必须是列表')
    if 'evidence' in value and any(not isinstance(item, str) for item in value['evidence']):
        raise ValueError('模型返回的 evidence 每项必须是字符串')
    if 'claims' in value and not isinstance(value['claims'], list):
        raise ValueError('模型返回的 claims 必须是列表')
    return value

call_attempts = []
def call_model(prompt):
    call_attempts.append('ZHIPUAI_API_KEY')
    raw = llm_call(prompt, max_tokens=700)
    return raw, 'ZHIPUAI_API_KEY'

def run_real_judge(sample, stepwise):
    template = STEPWISE_CALIBRATION_PROMPT if stepwise else DIRECT_CALIBRATION_PROMPT
    prompt = template.format(question=sample['question'], context=sample['context'], answer=sample['answer'])
    raw, key_source = call_model(prompt)
    parsed = parse_real_judge(raw)
    return {'status': '成功', 'raw': raw, 'key_source': key_source,
            'parsed': parsed, 'score': parsed['score']}

case = next(item for item in load_query_catalog() if item['id'] == 'contextual_cv_three_methods')
page18 = next(item['text'] for item in load_pdf_pages() if item['page'] == 18)
case_annotation = load_annotation(case['id'])
samples = [
    {
        'label': '有原文依据的校准样本',
        'question': case['query'],
        'context': page18,
        'answer': case_annotation['reference_answer'],
        'human_score': 2,
        'human_expectation': '完整且有原文依据',
    },
    {
        'label': '含明确错误结论的校准样本',
        'question': case['query'],
        'context': page18,
        'answer': '第 2.2 节只介绍留出法；交叉验证法和自助法属于别的章节，所以模型评估不需要它们。',
        'human_score': 0,
        'human_expectation': '主要结论错误，资料没有支持',
    },
]

real_results = []
print('真实校准模型：glm-4-flash；同一问题、同一页资料、两条回答；目标调用 4 次。')
for sample in samples:
    print('\n样本类型：', sample['label'])
    print('问题：', sample['question'])
    print('回答：', sample['answer'])
    print('人工预期：', sample['human_expectation'], '；人工分数：', sample['human_score'])
    for mode_name, stepwise in (('直接评分', False), ('逐项核对评分', True)):
        result = run_real_judge(sample, stepwise)
        real_results.append((sample, mode_name, result))
        print(mode_name, '状态：', result['status'], '；调用来源：', result['key_source'])
        parsed = result['parsed']
        print('模型分数：', result['score'], '；简短理由：', parsed['reason'])
        print('模型返回证据：', json.dumps(parsed.get('evidence', []), ensure_ascii=False))
        for index, item in enumerate(parsed.get('claims', []), start=1):
            if not isinstance(item, dict):
                raise ValueError('claims 中每项必须是对象')
            claim_text = item.get('claim', '')
            claim_evidence = json.dumps(item.get('evidence', []), ensure_ascii=False)
            claim_support = item.get('supported', '未说明')
            print(f'第 {index} 条结论：{claim_text}；证据：{claim_evidence}；资料支持：{claim_support}')
        print('与人工预期一致：', result['score'] == sample['human_score'])

success_count = sum(result['status'] == '成功' for _, _, result in real_results)
print('调用模型：glm-4-flash；实际调用尝试次数：', len(call_attempts), '；成功返回次数：', success_count)
if len(real_results) != 4 or success_count != 4:
    raise AssertionError('四次真实校准调用必须全部成功并保存结构化结果')
print('校准结论：两条人工预期和四次评分结果均已保存，可逐条复核；这不是 RAG 方法效果比较。')

真实校准模型：glm-4-flash；同一问题、同一页资料、两条回答；目标调用 4 次。

样本类型： 有原文依据的校准样本
问题： 第2.2节列出的三种模型评估办法分别叫什么？
回答： 第 2.2 节介绍留出法、交叉验证法和自助法。
人工预期： 完整且有原文依据 ；人工分数： 2


直接评分 状态： 成功 ；调用来源： ZHIPUAI_API_KEY
模型分数： 2 ；简短理由： 完整回应且主要结论都有资料依据
模型返回证据： ["留出法", "交叉验证法", "自助法"]
与人工预期一致： True


逐项核对评分 状态： 成功 ；调用来源： ZHIPUAI_API_KEY
模型分数： 2 ；简短理由： 文中明确提到了三种模型评估方法：留出法、交叉验证法和自助法。
模型返回证据： []
第 1 条结论：第 2.2 节介绍留出法；证据：["留出法"]；资料支持：True
第 2 条结论：第 2.2 节介绍交叉验证法；证据：["交叉验证法"]；资料支持：True
第 3 条结论：第 2.2 节介绍自助法；证据：["自助法"]；资料支持：True
与人工预期一致： True

样本类型： 含明确错误结论的校准样本
问题： 第2.2节列出的三种模型评估办法分别叫什么？
回答： 第 2.2 节只介绍留出法；交叉验证法和自助法属于别的章节，所以模型评估不需要它们。
人工预期： 主要结论错误，资料没有支持 ；人工分数： 0


直接评分 状态： 成功 ；调用来源： ZHIPUAI_API_KEY
模型分数： 0 ；简短理由： 主要结论错误
模型返回证据： ["资料中明确提到第2.2节介绍了三种模型评估方法：留出法、交叉验证法、自助法。"]
与人工预期一致： True


逐项核对评分 状态： 成功 ；调用来源： ZHIPUAI_API_KEY
模型分数： 2 ；简短理由： 有两条结论有证据支持，但其中一条结论的关联性较弱。
模型返回证据： []
第 1 条结论：第 2.2 节只介绍留出法；证据：["留出法", "留出法由于操作简单，因此最常用"]；资料支持：True
第 2 条结论：交叉验证法属于别的章节；证据：["交叉验证法常用于对比同一算法的不同参数配置之间的效果，以及对比不同算法之间的效果"]；资料支持：True
第 3 条结论：自助法属于别的章节；证据：["自助法常用于集成学习（详见“西瓜书”第 8 章的 8.2 节和 8.3 节）产生基分类器"]；资料支持：True
第 4 条结论：模型评估不需要交叉验证法和自助法；证据：["留出法和自助法简单易懂，在此不再赘述", "自助法常用于集成学习"]；资料支持：False
第 5 条结论：模型评估不需要交叉验证法；证据：["交叉验证法常用于对比同一算法的不同参数配置之间的效果，以及对比不同算法之间的效果"]；资料支持：False
与人工预期一致： False
调用模型：glm-4-flash；实际调用尝试次数： 4 ；成功返回次数： 4
校准结论：两条人工预期和四次评分结果均已保存，可逐条复核；这不是 RAG 方法效果比较。


## 发现误判后怎样修正：语义关系和引用核验

改进的逐项评分要求每条结论标为“资料支持（supported）”“资料反驳（contradicted）”或“资料未提及（not_found）”。引用必须是资料中的原文短语；出现相同词语不代表支持，评审器还要判断资料和回答的语义方向。只要主要结论被资料直接反驳，总分就不能是 2。

代码会逐条检查引用短语（忽略空白后）是否真的出现在资料中；解析错误、非法关系或找不到的短语都会直接抛错，不会静默算作支持。改进复查仍只把问题、资料和回答传给模型，人工分数只在模型返回后用于对照。

In [4]:
import re

IMPROVED_STEPWISE_PROMPT = """你是严格的资料依据评审员，不是回答者。只根据问题、资料和回答判断，不要使用参考答案或人工分数。
先把回答拆成不超过 5 条主要结论。每条结论必须填写 relation：supported（资料支持）、contradicted（资料反驳）或 not_found（资料未提及）三者之一；证据只能逐字摘自资料中的短语，不得用你的改写。出现相关词语不等于支持，必须判断资料和回答的语义方向。若资料直接反驳主要结论，relation 必须为 contradicted，且总分绝对不能是 2。relation=not_found 时 evidence 必须严格是 ["NOTHING_FOUND"]，不能写解释性句子。
总分只能是整数 0、1 或 2：0=主要结论错误或被反驳，1=部分正确或部分有据，2=完整且主要结论均有据。只输出 JSON：{{\"claims\": [{{\"claim\": \"结论\", \"relation\": \"supported|contradicted|not_found\", \"evidence\": [\"原文短语或 NOTHING_FOUND\"]}}], \"score\": 0|1|2, \"reason\": \"一句话\"}}。
问题：{question}\n资料：{context}\n回答：{answer}"""

RELATION_LABELS = {'supported': '资料支持', 'contradicted': '资料反驳', 'not_found': '资料未提及'}

def parse_improved_judge(raw):
    value = parse_real_judge(raw)
    claims = value.get('claims')
    if not isinstance(claims, list) or not claims:
        raise ValueError('逐项结果缺少 claims 列表')
    for item in claims:
        if not isinstance(item, dict) or set(item) != {'claim', 'relation', 'evidence'}:
            raise ValueError('逐项结果每条 claim 必须严格包含 claim/relation/evidence')
        if not isinstance(item['claim'], str) or not item['claim'].strip():
            raise ValueError('逐项结果的 claim 必须是非空字符串')
        if item['relation'] not in RELATION_LABELS:
            raise ValueError('逐项结果的 relation 不在允许范围内')
        if not isinstance(item['evidence'], list):
            raise ValueError('逐项结果的 evidence 必须是列表')
        if any(not isinstance(phrase, str) for phrase in item['evidence']):
            raise ValueError('逐项结果的 evidence 每项必须是字符串')
        if any(not phrase.strip() for phrase in item['evidence']):
            raise ValueError('逐项结果的 evidence 每项必须是非空字符串')
        if item['relation'] == 'not_found' and item['evidence'] != ['NOTHING_FOUND']:
            raise ValueError('not_found 的 evidence 必须严格为 [NOTHING_FOUND]')
        if item['relation'] in {'supported', 'contradicted'} and not item['evidence']:
            raise ValueError('supported/contradicted 必须提供原文 evidence')
    return value

def normalize_for_evidence(value):
    return re.sub(r'\s+', '', str(value or ''))

def verify_evidence(parsed, context):
    normalized_context = normalize_for_evidence(context)
    rows = []
    for index, item in enumerate(parsed['claims'], start=1):
        relation = item['relation']
        phrases = [normalize_for_evidence(phrase) for phrase in item['evidence']]
        if relation == 'not_found':
            if item['evidence'] != ['NOTHING_FOUND']:
                raise ValueError(f'第 {index} 条 not_found evidence 必须严格为 NOTHING_FOUND')
            found = []
            missing = []
            valid = True
            status = '未引用（模型声明资料未提及）'
        else:
            found = [phrase for phrase in phrases if phrase in normalized_context]
            missing = [phrase for phrase in phrases if phrase not in normalized_context]
            if not phrases or missing:
                raise ValueError(f'第 {index} 条 {relation} 的 evidence 未逐字命中资料：{missing!r}')
            valid = True
            status = '引用全部在资料中找到'
        rows.append({'index': index, 'claim': item.get('claim', ''), 'relation': relation,
                     'evidence': item['evidence'], 'found': found, 'missing': missing,
                     'valid': valid, 'status': status})
    if any(row['relation'] == 'contradicted' for row in rows) and parsed['score'] == 2:
        raise ValueError('存在 contradicted 主要结论时 score 不能为 2')
    return rows, True

def run_improved_stepwise(sample):
    prompt = IMPROVED_STEPWISE_PROMPT.format(question=sample['question'], context=sample['context'], answer=sample['answer'])
    raw, key_source = call_model(prompt)
    parsed = parse_improved_judge(raw)
    rows, contradiction_guard = verify_evidence(parsed, sample['context'])
    return {'status': '成功', 'raw': raw, 'key_source': key_source,
            'parsed': parsed, 'score': parsed['score'], 'rows': rows,
            'contradiction_guard': contradiction_guard,
            'evidence_verifiable': all(row['valid'] for row in rows)}

improved_start_attempts = len(call_attempts)
improved_results = []
print('改进逐项复查：只增加同一两条样本的 2 次 glm-4-flash 调用；模型输入只有问题、资料和回答。')
for sample in samples:
    result = run_improved_stepwise(sample)
    improved_results.append((sample, result))
    print('\n样本类型：', sample['label'], '；调用来源：', result.get('key_source') or '无', '；状态：', result['status'])
    parsed = result['parsed']
    print('模型分数：', result['score'], '；简短理由：', parsed['reason'])
    for row in result['rows']:
        missing_text = json.dumps(row['missing'], ensure_ascii=False) if row['missing'] else '无'
        print(f"第 {row['index']} 条结论：{row['claim']}；关系：{RELATION_LABELS[row['relation']]}；引用：{json.dumps(row['evidence'], ensure_ascii=False)}；代码核验：{row['status']}；未命中引用：{missing_text}")
    print('引用全部可核验：', result['evidence_verifiable'], '；证据与反驳约束均通过：', result['contradiction_guard'])
    print('与人工预期一致：', result['score'] == sample['human_score'])

improved_correct = sum(result['status'] == '成功' and result['score'] == sample['human_score']
                       for sample, result in improved_results)
improved_evidence_verifiable = sum(result['status'] == '成功' and result['evidence_verifiable']
                                   for _, result in improved_results)
improved_calls = len(call_attempts) - improved_start_attempts
if improved_calls != 2 or improved_correct != 2 or improved_evidence_verifiable != 2:
    raise AssertionError('两次改进复查必须成功解析、引用全部可核验且分数符合固定校准预期')
print('\n比较：原始逐项评分的最终分数正确 1/2；改进后最终分数正确', f'{improved_correct}/2，',
      '两条结果的引用全部可核验。')
print('额外调用：', improved_calls, '次 glm-4-flash；相对原始记录新增这些请求，未读取计费价格。')
print('限制（历史复查）：只有两条历史样本，不能证明改进后的 Prompt 对一般任务普遍更好。')

改进逐项复查：只增加同一两条样本的 2 次 glm-4-flash 调用；模型输入只有问题、资料和回答。



样本类型： 有原文依据的校准样本 ；调用来源： ZHIPUAI_API_KEY ；状态： 成功
模型分数： 2 ；简短理由： 所有主要结论均有据
第 1 条结论：第 2.2 节介绍留出法；关系：资料支持；引用：["留出法由于操作简单，因此最常用"]；代码核验：引用全部在资料中找到；未命中引用：无
第 2 条结论：第 2.2 节介绍交叉验证法；关系：资料支持；引用：["交叉验证法常用于对比同一算法的不同参数配置之间的效果，以及对比不同算法之间的效果"]；代码核验：引用全部在资料中找到；未命中引用：无
第 3 条结论：第 2.2 节介绍自助法；关系：资料支持；引用：["自助法常用于集成学习（详见“西瓜书”第 8 章的 8.2 节和 8.3 节）产生基分类器"]；代码核验：引用全部在资料中找到；未命中引用：无
引用全部可核验： True ；证据与反驳约束均通过： True
与人工预期一致： True



样本类型： 含明确错误结论的校准样本 ；调用来源： ZHIPUAI_API_KEY ；状态： 成功
模型分数： 0 ；简短理由： 回答中的结论与资料中的信息不符，资料明确指出第 2.2 节介绍了三种模型评估方法，而回答中提到不需要这些方法，资料中并未提及这一点。
第 1 条结论：第 2.2 节只介绍留出法；关系：资料反驳；引用：["本节介绍了3 种模型评估方法：留出法、交叉验证法、自助法"]；代码核验：引用全部在资料中找到；未命中引用：无
第 2 条结论：交叉验证法和自助法属于别的章节；关系：资料未提及；引用：["NOTHING_FOUND"]；代码核验：未引用（模型声明资料未提及）；未命中引用：无
第 3 条结论：所以模型评估不需要它们；关系：资料未提及；引用：["NOTHING_FOUND"]；代码核验：未引用（模型声明资料未提及）；未命中引用：无
引用全部可核验： True ；证据与反驳约束均通过： True
与人工预期一致： True

比较：原始逐项评分的最终分数正确 1/2；改进后最终分数正确 2/2， 两条结果的引用全部可核验。
额外调用： 2 次 glm-4-flash；相对原始记录新增这些请求，未读取计费价格。
限制（历史复查）：只有两条历史样本，不能证明改进后的 Prompt 对一般任务普遍更好。


## 扩展校准集：10 条真实样本

上一段的两条样本用于保留第一次误判和修正过程；这里把校准集扩大到 10 条并增加错误类型。每条记录都从 canonical `evidence.jsonl` 读取现有 `evidence_id`、页码和原文 `quote`，再写出一个教学用待审回答。`label_source=tutorial_author_annotation` 是仓库内的教学预设标签，且明确记录 `human_verified=false`；它不是独立人工语义核验、外部人工研究、众包结果或独立金标准。

本单元的分数协议是：`2` 表示完整回答；如果资料不足，明确说资料不能回答也算完成任务；`1` 表示只覆盖部分要求；`0` 表示主要结论错误、无依据、引用错配，或面对应该拒答的问题仍然编造答案。除了分数，还要求模型显式返回 `needs_refusal`、`answer_action` 和 `citation_ok`，这样“应该拒答”和“引用错配”不会被一个分数掩盖。

下面每条只发出一次真实请求。原始返回先原样记录，再解析；解析失败不会重试、不会 fallback，也不会填入伪造结果。

### 1. 先检查样本和教学预设

先运行下一格，核对 10 条待审回答、canonical evidence ID、错误类型和 `human_verified=false`。这里不调用模型。


In [5]:
import json
import re
import unicodedata
from collections import Counter, defaultdict

from common.dataset import load_search_evidence
from common.nontraining_utils import RAG_LLM_MODEL

EXPANDED_MODEL = "glm-4-flash"
assert RAG_LLM_MODEL == EXPANDED_MODEL, "校准集必须固定使用 glm-4-flash"

query_rows = {row["id"]: row for row in load_query_catalog()}
evidence_rows = load_search_evidence()
evidence_by_id = {row["evidence_id"]: row for row in evidence_rows}

def compact_ws(value):
    return re.sub(r"\s+", "", str(value or ""))

def quote_match_text(value):
    # 与 Agentic 契约一致：NFKC 后只去空白，保留标点、数字和运算符。
    normalized = unicodedata.normalize("NFKC", str(value or "")).lower()
    return "".join(character for character in normalized if not character.isspace())

def quote_overlaps_evidence(quote, evidence_ids):
    candidate = quote_match_text(quote)
    if not candidate:
        return False
    for evidence_id in evidence_ids:
        canonical = quote_match_text(evidence_by_id[evidence_id]["quote"])
        if candidate in canonical:
            return True
    return False

def evidence_context(evidence_ids):
    if not evidence_ids:
        raise ValueError("每条校准样本至少要绑定一个 canonical evidence")
    blocks = []
    for evidence_id in evidence_ids:
        row = evidence_by_id.get(evidence_id)
        if row is None:
            raise KeyError(f"canonical evidence 不存在：{evidence_id}")
        blocks.append(
            f"[evidence_id={evidence_id}; page={row['page']}]\n{row['quote']}"
        )
    return "\n\n".join(blocks)

def make_calibration_sample(
    sample_id,
    category,
    query_id,
    answer,
    context_evidence_ids,
    expected_score,
    expected_needs_refusal,
    expected_answer_action,
    expected_citation_ok=True,
    expected_support_evidence_ids=None,
    scenario=None,
):
    if query_id not in query_rows:
        raise KeyError(f"问题集不存在：{query_id}")
    context_ids = list(context_evidence_ids)
    support_ids = list(expected_support_evidence_ids or [])
    if not set(support_ids).issubset(context_ids):
        raise ValueError(f"{sample_id} 的期望支持 evidence 不在上下文中")
    return {
        "sample_id": sample_id,
        "category": category,
        "scenario": scenario or category,
        "query_id": query_id,
        "question": query_rows[query_id]["query"],
        "context_evidence_ids": context_ids,
        "evidence_binding": [
            {
                "evidence_id": evidence_id,
                "page": evidence_by_id[evidence_id]["page"],
                "quote": evidence_by_id[evidence_id]["quote"],
            }
            for evidence_id in context_ids
        ],
        "answer": answer,
        "human_expectation": {
            "score": expected_score,
            "needs_refusal": expected_needs_refusal,
            "answer_action": expected_answer_action,
            "citation_ok": expected_citation_ok,
        },
        "label_source": "tutorial_author_annotation",
        "human_verified": False,
        "expected_support_evidence_ids": support_ids,
        "context": evidence_context(context_ids),
    }

calibration_samples = [
    make_calibration_sample(
        "cal-01",
        "correct",
        "model_evaluation_purpose",
        "模型评估与选择用于评估模型优劣，并选择适合业务场景的模型。",
        ["evi_91d0b1e7d0ff"],
        2,
        False,
        "answer",
        expected_support_evidence_ids=["evi_91d0b1e7d0ff"],
        scenario="完整且有据",
    ),
    make_calibration_sample(
        "cal-02",
        "correct",
        "model_evaluation_and_macro_micro",
        "“模型评估与选择”用于评估模型优劣并选择适合业务场景的模型；类别不平衡时，宏平均平等看待类别，微平均考虑各类样本数量，数量较多的类别可能主导微平均结果。",
        ["evi_91d0b1e7d0ff", "evi_ffd64ad64243"],
        2,
        False,
        "answer",
        expected_support_evidence_ids=["evi_91d0b1e7d0ff", "evi_ffd64ad64243"],
        scenario="跨页完整回答",
    ),
    make_calibration_sample(
        "cal-03",
        "partial",
        "cross_validation_reliability",
        "交叉验证法会多次换不同的子集作测试集，让所有样本至少做一次测试。",
        ["evi_9a292ab3d933", "evi_833c8fa10a30"],
        1,
        False,
        "answer",
        expected_support_evidence_ids=["evi_9a292ab3d933"],
        scenario="遗漏单次留出法偶然性",
    ),
    make_calibration_sample(
        "cal-04",
        "partial",
        "newton_hessian_cost",
        "牛顿法每次迭代需要求 Hessian 矩阵的逆，计算量通常较大。",
        ["evi_5b020fc7351f", "evi_99aa3c418c53"],
        1,
        False,
        "answer",
        expected_support_evidence_ids=["evi_5b020fc7351f"],
        scenario="遗漏全局最优保证边界",
    ),
    make_calibration_sample(
        "cal-05",
        "unsupported",
        "macro_micro_imbalance",
        "宏平均会按每个类别的样本数量加权，微平均则平等看待每个类别；类别极度不平衡时，宏平均会被数量较多的类别主导。",
        ["evi_ffd64ad64243"],
        0,
        False,
        "answer",
        expected_citation_ok=False,
        expected_support_evidence_ids=[],
        scenario="没有显式引用，但主要结论被 context 直接反驳",
    ),
    make_calibration_sample(
        "cal-06",
        "unsupported",
        "model_selection_no_absolute_best",
        "存在一个在所有任务上都绝对最好的机器学习算法。",
        ["evi_32c0b230bb7c"],
        0,
        False,
        "answer",
        expected_citation_ok=False,
        expected_support_evidence_ids=[],
        scenario="没有显式引用，但主要结论被原文直接反驳",
    ),
    make_calibration_sample(
        "cal-07",
        "citation_mismatch",
        "model_evaluation_purpose",
        "模型评估与选择用于评估模型优劣，并选择适合业务场景的模型。[evi_ffd64ad64243]",
        ["evi_91d0b1e7d0ff", "evi_ffd64ad64243"],
        0,
        False,
        "answer",
        expected_citation_ok=False,
        expected_support_evidence_ids=["evi_91d0b1e7d0ff"],
        scenario="结论基本正确但显式引用了宏/微平均段落",
    ),
    make_calibration_sample(
        "cal-08",
        "citation_mismatch",
        "cross_validation_reliability",
        "交叉验证比单次留出法更可靠，因为它会多次换测试子集、让所有样本至少测试一次。[evi_91d0b1e7d0ff]",
        ["evi_9a292ab3d933", "evi_833c8fa10a30", "evi_91d0b1e7d0ff"],
        0,
        False,
        "answer",
        expected_citation_ok=False,
        expected_support_evidence_ids=["evi_9a292ab3d933", "evi_833c8fa10a30"],
        scenario="结论基本正确但显式引用了模型评估目的段落",
    ),
    make_calibration_sample(
        "cal-09",
        "should_refuse",
        "book_evidence_boundary",
        "资料中没有 CUDA 版本建议，不能仅依据这份南瓜书回答。",
        ["evi_91d0b1e7d0ff"],
        2,
        True,
        "refuse",
        expected_citation_ok=True,
        expected_support_evidence_ids=[],
        scenario="资料不足时正确拒答",
    ),
    make_calibration_sample(
        "cal-10",
        "unsupported",
        "book_evidence_mnist",
        "这份资料建议使用 MNIST 的 60000 个训练样本，并且测试集有 10000 个样本。",
        ["evi_718a2d52ed5c"],
        0,
        True,
        "answer",
        expected_citation_ok=False,
        expected_support_evidence_ids=[],
        scenario="资料不足却实际编造外部数据；问题应拒答，但待审回答的实际动作是 answer",
    ),
]

assert len(calibration_samples) == 10
assert len({sample["sample_id"] for sample in calibration_samples}) == 10
for sample in calibration_samples:
    assert sample["label_source"] == "tutorial_author_annotation"
    assert sample["human_verified"] is False
    assert sample["evidence_binding"]
    assert all(
        binding["quote"] == evidence_by_id[binding["evidence_id"]]["quote"]
        for binding in sample["evidence_binding"]
    )

for sample in calibration_samples:
    print(
        sample["sample_id"],
        sample["category"],
        "query_id=" + sample["query_id"],
        "answer=" + sample["answer"],
        "evidence_ids=" + json.dumps(sample["context_evidence_ids"], ensure_ascii=False),
        "human_expectation=" + json.dumps(sample["human_expectation"], ensure_ascii=False),
    )



cal-01 correct query_id=model_evaluation_purpose answer=模型评估与选择用于评估模型优劣，并选择适合业务场景的模型。 evidence_ids=["evi_91d0b1e7d0ff"] human_expectation={"score": 2, "needs_refusal": false, "answer_action": "answer", "citation_ok": true}
cal-02 correct query_id=model_evaluation_and_macro_micro answer=“模型评估与选择”用于评估模型优劣并选择适合业务场景的模型；类别不平衡时，宏平均平等看待类别，微平均考虑各类样本数量，数量较多的类别可能主导微平均结果。 evidence_ids=["evi_91d0b1e7d0ff", "evi_ffd64ad64243"] human_expectation={"score": 2, "needs_refusal": false, "answer_action": "answer", "citation_ok": true}
cal-03 partial query_id=cross_validation_reliability answer=交叉验证法会多次换不同的子集作测试集，让所有样本至少做一次测试。 evidence_ids=["evi_9a292ab3d933", "evi_833c8fa10a30"] human_expectation={"score": 1, "needs_refusal": false, "answer_action": "answer", "citation_ok": true}
cal-04 partial query_id=newton_hessian_cost answer=牛顿法每次迭代需要求 Hessian 矩阵的逆，计算量通常较大。 evidence_ids=["evi_5b020fc7351f", "evi_99aa3c418c53"] human_expectation={"score": 1, "needs_refusal": false, "answer_action": "answer", "citation

### 2. 再定义评分协议和严格解析

这一格只定义 prompt、字段约束和逐条原文核验。可以先阅读 `needs_refusal` 与 `answer_action` 的区别，再继续批量实验。


In [6]:
EXPANDED_CALIBRATION_PROMPT = """你是严格的资料依据、引用和拒答审计员，不是回答者。只能使用给定的 question、context 和 answer，不要使用外部常识、参考答案或人工标签。
评分规则：
- score=2：完整回答问题且主要结论有资料支持；如果问题超出 context，明确说明资料不足并拒答，也算完成任务。
- score=1：只回答了部分要求，或主要结论只有部分得到资料支持。
- score=0：主要结论错误、被资料反驳、资料没有依据，显式引用与结论错配，或问题应该拒答却编造答案。
如果 answer 中出现 [evi_...] 引用标记，必须检查该 ID 对应的 quote 是否直接支持所引用的结论；错配时 citation_ok=false 且 score 必须为 0。
needs_refusal 表示仅根据 context 判断该问题是否应该拒答；answer_action 表示 answer 实际是在回答（answer）还是拒答（refuse）。没有显式引用标记时，citation_ok 表示回答中的主要结论是否能由 context 直接支持。
把 answer 拆成不超过 5 条主要结论。每条 claim 必须填写 relation：supported、contradicted 或 not_found；evidence_ids 只能来自 context 中出现的 ID；evidence 必须是 context 中逐字出现的 quote，not_found 时严格写 ["NOTHING_FOUND"]。如果资料直接反驳主要结论，不能给 score=2。
只输出一个 JSON 对象，不要 Markdown 围栏、解释文字或额外字段：
{{"claims":[{{"claim":"结论","relation":"supported|contradicted|not_found","evidence_ids":["evi_..."],"evidence":["原文 quote 或 NOTHING_FOUND"]}}],"score":0|1|2,"reason":"一句话","citation_ok":true|false,"needs_refusal":true|false,"answer_action":"answer|refuse","cited_evidence_ids":["evi_..."]}}
question：{question}
context：
{context}
answer：
{answer}"""

def parse_expanded_judge(raw, context_ids):
    text = str(raw or "").strip()
    fence = chr(96) * 3
    if text.startswith(fence + "json") and text.endswith(fence):
        text = text[len(fence + "json"):-len(fence)].strip()
    value = json.loads(text)
    required = {
        "claims",
        "score",
        "reason",
        "citation_ok",
        "needs_refusal",
        "answer_action",
        "cited_evidence_ids",
    }
    if not isinstance(value, dict) or set(value) != required:
        raise ValueError("模型返回字段必须严格为 claims/score/reason/citation_ok/needs_refusal/answer_action/cited_evidence_ids")
    if isinstance(value["score"], bool) or value["score"] not in (0, 1, 2):
        raise ValueError("score 必须是 0、1 或 2")
    if not isinstance(value["reason"], str) or not value["reason"].strip():
        raise ValueError("reason 必须是非空字符串")
    for field in ("citation_ok", "needs_refusal"):
        if not isinstance(value[field], bool):
            raise ValueError(f"{field} 必须是布尔值")
    if value["answer_action"] not in {"answer", "refuse"}:
        raise ValueError("answer_action 必须是 answer 或 refuse")
    if (
        not isinstance(value["cited_evidence_ids"], list)
        or any(
            not isinstance(evidence_id, str)
            or evidence_id not in context_ids
            for evidence_id in value["cited_evidence_ids"]
        )
    ):
        raise ValueError("cited_evidence_ids 必须来自 context")
    if not isinstance(value["claims"], list) or not value["claims"]:
        raise ValueError("claims 必须是非空列表")
    evidence_issues = []
    for index, claim in enumerate(value["claims"], start=1):
        if not isinstance(claim, dict) or set(claim) != {
            "claim", "relation", "evidence_ids", "evidence"
        }:
            raise ValueError(f"第 {index} 条 claim 字段不完整")
        if not isinstance(claim["claim"], str) or not claim["claim"].strip():
            raise ValueError(f"第 {index} 条 claim 不能为空")
        if claim["relation"] not in {"supported", "contradicted", "not_found"}:
            raise ValueError(f"第 {index} 条 relation 非法")
        if (
            not isinstance(claim["evidence_ids"], list)
            or any(
                not isinstance(evidence_id, str)
                or evidence_id not in context_ids
                for evidence_id in claim["evidence_ids"]
            )
        ):
            raise ValueError(f"第 {index} 条 evidence_ids 不在 context")
        if not isinstance(claim["evidence"], list) or not claim["evidence"] or any(
            not isinstance(quote, str) or not quote.strip()
            for quote in claim["evidence"]
        ):
            raise ValueError(f"第 {index} 条 evidence 必须是非空字符串列表")
        if claim["relation"] == "not_found":
            if claim["evidence"] != ["NOTHING_FOUND"]:
                raise ValueError(f"第 {index} 条 not_found 必须使用 NOTHING_FOUND")
        else:
            if not claim["evidence_ids"]:
                raise ValueError(f"第 {index} 条 {claim['relation']} 缺少 evidence_id")
            invalid_quotes = [
                quote for quote in claim["evidence"]
                if not quote_overlaps_evidence(quote, claim["evidence_ids"])
            ]
            if invalid_quotes:
                evidence_issues.append(
                    f"第 {index} 条有 {len(invalid_quotes)} 个 evidence 摘录不是所列 canonical quote 的完整连续子串"
                )
    if any(claim["relation"] == "contradicted" for claim in value["claims"]) and value["score"] == 2:
        raise ValueError("存在 contradicted claim 时 score 不能是 2")
    if value["needs_refusal"] and value["answer_action"] == "answer" and value["score"] == 2:
        raise ValueError("应该拒答却回答时 score 不能是 2")
    value["evidence_verifiable"] = not evidence_issues
    value["evidence_issues"] = evidence_issues
    return value

def classify_prediction(parsed):
    if parsed["needs_refusal"] and parsed["answer_action"] == "refuse":
        return "should_refuse"
    if not parsed["citation_ok"]:
        return "citation_mismatch"
    if parsed["score"] == 2:
        return "correct"
    if parsed["score"] == 1:
        return "partial"
    return "unsupported"



### 3. 运行固定批次并保留结果

这一格才发出 10 次真实请求，逐条保留 raw、parsed 和错误记录；统计在下一格单独计算。任何调用或解析失败都会让本次校准直接失败。


In [7]:
expanded_call_attempts = []
expanded_results = []

def call_expanded_model(prompt):
    expanded_call_attempts.append({
        "model": EXPANDED_MODEL,
        "max_retries": 0,
        "key_source": "ZHIPUAI_API_KEY",
    })
    # common.nontraining_utils.llm_call 创建 SDK client 时固定 max_retries=0；
    # 本循环每个样本只调用一次，异常直接记录，不进行应用层重试或 fallback。
    return llm_call(prompt, max_tokens=900)

print("\n扩展校准：固定模型", EXPANDED_MODEL, "；每条一次调用；SDK max_retries=0；目标 10 次。")
for sample in calibration_samples:
    prompt = EXPANDED_CALIBRATION_PROMPT.format(
        question=sample["question"],
        context=sample["context"],
        answer=sample["answer"],
    )
    record = {
        "sample_id": sample["sample_id"],
        "category": sample["category"],
        "query_id": sample["query_id"],
        "question": sample["question"],
        "answer": sample["answer"],
        "context_evidence_ids": sample["context_evidence_ids"],
        "human_expectation": sample["human_expectation"],
        "raw": None,
        "status": "调用失败",
        "parsed": None,
        "error": None,
    }
    try:
        raw = call_expanded_model(prompt)
        record["raw"] = raw
        record["status"] = "成功"
        try:
            record["parsed"] = parse_expanded_judge(raw, sample["context_evidence_ids"])
        except Exception as error:
            record["status"] = "解析失败"
            record["error"] = f"{type(error).__name__}: {error}"
    except Exception as error:
        record["error"] = f"{type(error).__name__}: {error}"
    expanded_results.append(record)
    print(
        f"\n{sample['sample_id']} [{sample['category']}] "
        f"status={record['status']} evidence_ids={json.dumps(sample['context_evidence_ids'], ensure_ascii=False)}"
    )
    print("问题：", sample["question"])
    print("回答：", sample["answer"])
    print("教程预设期望（human_verified=false）：", json.dumps(sample["human_expectation"], ensure_ascii=False))
    if record["raw"] is not None:
        # raw 是服务返回的原始文字；这里不改写、不补造。
        print("原始模型返回：", json.dumps(record["raw"], ensure_ascii=False))
    if record["parsed"] is not None:
        print("结构化判断：", json.dumps(record["parsed"], ensure_ascii=False))
        print("canonical quote 可核验：", record["parsed"]["evidence_verifiable"],
              "问题：", record["parsed"]["evidence_issues"] or "无")
    if record["error"] is not None:
        print("错误记录：", record["error"])

failed_results = [record for record in expanded_results if record["parsed"] is None]
if failed_results:
    raise RuntimeError('扩展校准存在调用或解析失败，禁止用子集继续计分：' + json.dumps(
        [{"sample_id": row["sample_id"], "status": row["status"], "error": row["error"]} for row in failed_results],
        ensure_ascii=False,
    ))



扩展校准：固定模型 glm-4-flash ；每条一次调用；SDK max_retries=0；目标 10 次。



cal-01 [correct] status=成功 evidence_ids=["evi_91d0b1e7d0ff"]
问题： 为什么要做模型评估和选择？
回答： 模型评估与选择用于评估模型优劣，并选择适合业务场景的模型。
教程预设期望（human_verified=false）： {"score": 2, "needs_refusal": false, "answer_action": "answer", "citation_ok": true}
原始模型返回： "```json\n{\n  \"claims\": [\n    {\n      \"claim\": \"模型评估与选择用于评估模型优劣\",\n      \"relation\": \"supported\",\n      \"evidence_ids\": [\"evi_91d0b1e7d0ff\"],\n      \"evidence\": [\"“模型评估与选择”所述，讲述的是如何评估模型的优劣和选择最适合自己业务场景的模型\"]\n    },\n    {\n      \"claim\": \"模型评估与选择用于选择适合业务场景的模型\",\n      \"relation\": \"supported\",\n      \"evidence_ids\": [\"evi_91d0b1e7d0ff\"],\n      \"evidence\": [\"“模型评估与选择”所述，讲述的是如何评估模型的优劣和选择最适合自己业务场景的模型\"]\n    }\n  ],\n  \"score\": 2,\n  \"reason\": \"主要结论有资料支持\",\n  \"citation_ok\": true,\n  \"needs_refusal\": false,\n  \"answer_action\": \"answer\",\n  \"cited_evidence_ids\": [\"evi_91d0b1e7d0ff\"]\n}\n```"
结构化判断： {"claims": [{"claim": "模型评估与选择用于评估模型优劣", "relation": "supported", "evidence_ids": ["evi_91d0b1e7d0ff"], "evi


cal-02 [correct] status=成功 evidence_ids=["evi_91d0b1e7d0ff", "evi_ffd64ad64243"]
问题： “模型评估与选择”是什么？为什么类别不平衡时要区分宏平均和微平均？
回答： “模型评估与选择”用于评估模型优劣并选择适合业务场景的模型；类别不平衡时，宏平均平等看待类别，微平均考虑各类样本数量，数量较多的类别可能主导微平均结果。
教程预设期望（human_verified=false）： {"score": 2, "needs_refusal": false, "answer_action": "answer", "citation_ok": true}
原始模型返回： "```json\n{\n  \"claims\": [\n    {\n      \"claim\": \"“模型评估与选择”用于评估模型优劣并选择适合业务场景的模型\",\n      \"relation\": \"supported\",\n      \"evidence_ids\": [\"evi_91d0b1e7d0ff\"],\n      \"evidence\": [\"“模型评估与选择”所述，讲述的是如何评估模型的优劣和选择最适合自己业务场景的模型\"]\n    },\n    {\n      \"claim\": \"类别不平衡时，宏平均平等看待类别\",\n      \"relation\": \"supported\",\n      \"evidence_ids\": [\"evi_ffd64ad64243\"],\n      \"evidence\": [\"因此，‘宏’没有考虑每个类别下的的样本数量，所以平等看待每个 类别\"]\n    },\n    {\n      \"claim\": \"微平均考虑各类样本数量\",\n      \"relation\": \"supported\",\n      \"evidence_ids\": [\"evi_ffd64ad64243\"],\n      \"evidence\": [\"而“微”则考虑到了每个类别的样本数量\"]\n    },\n    {\n      \"claim\": \"数量较多的类别可能主导微平均结果\


cal-03 [partial] status=成功 evidence_ids=["evi_9a292ab3d933", "evi_833c8fa10a30"]
问题： 交叉验证法为什么比单次留出法更可靠？
回答： 交叉验证法会多次换不同的子集作测试集，让所有样本至少做一次测试。
教程预设期望（human_verified=false）： {"score": 1, "needs_refusal": false, "answer_action": "answer", "citation_ok": true}
原始模型返回： "```json\n{\n  \"claims\": [\n    {\n      \"claim\": \"交叉验证法会多次换不同的子集作测试集\",\n      \"relation\": \"supported\",\n      \"evidence_ids\": [\"evi_9a292ab3d933\", \"evi_833c8fa10a30\"],\n      \"evidence\": [\"从以上的举例可以看出，交叉验证法本质上是在进行多次留出法，且每次都换不同的子集做测试集， 最终让所有样本均至少做1 次测试样本。\", \"这样做的理由其实很简单，因为一般的留出法只会划分出1 组 训练集和测试集，仅依靠1 组训练集和测试集去对比不同算法之间的效果显然不够置信，偶然性太强， 因此要想基于固定的数据集产生多组不同的训练集和测试集，则只有进行多次划分，每次采用不同的子集 作为测试集，也即为交叉验证法。\"]\n    }\n  ],\n  \"score\": 2,\n  \"reason\": \"回答中提到的交叉验证法多次换不同的子集作测试集与资料中的描述一致。\",\n  \"citation_ok\": true,\n  \"needs_refusal\": false,\n  \"answer_action\": \"answer\",\n  \"cited_evidence_ids\": [\"evi_9a292ab3d933\", \"evi_833c8fa10a30\"]\n}\n```"
结构化判断： {"claims": [{"claim": "交叉验证法会多次换不同的子集作测试集", "relation


cal-04 [partial] status=成功 evidence_ids=["evi_5b020fc7351f", "evi_99aa3c418c53"]
问题： 牛顿法收敛快，但计算 Hessian 有什么代价？它是否一定得到全局最优解？
回答： 牛顿法每次迭代需要求 Hessian 矩阵的逆，计算量通常较大。
教程预设期望（human_verified=false）： {"score": 1, "needs_refusal": false, "answer_action": "answer", "citation_ok": true}
原始模型返回： "```json\n{\n  \"claims\": [\n    {\n      \"claim\": \"牛顿法每次迭代需要求 Hessian 矩阵的逆。\",\n      \"relation\": \"supported\",\n      \"evidence_ids\": [\"evi_5b020fc7351f\"],\n      \"evidence\": [\"通过上述推导可知，牛顿法每次迭代时需要求解Hessian 矩阵的逆矩阵，该步骤计算量通常较大\"]\n    }\n  ],\n  \"score\": 1,\n  \"reason\": \"主要结论得到资料支持，但未涵盖所有问题。\",\n  \"citation_ok\": true,\n  \"needs_refusal\": false,\n  \"answer_action\": \"answer\",\n  \"cited_evidence_ids\": [\"evi_5b020fc7351f\"]\n}\n```"
结构化判断： {"claims": [{"claim": "牛顿法每次迭代需要求 Hessian 矩阵的逆。", "relation": "supported", "evidence_ids": ["evi_5b020fc7351f"], "evidence": ["通过上述推导可知，牛顿法每次迭代时需要求解Hessian 矩阵的逆矩阵，该步骤计算量通常较大"]}], "score": 1, "reason": "主要结论得到资料支持，但未涵盖所有问题。", "citation_ok": true, 

...


cal-06 [unsupported] status=成功 evidence_ids=["evi_32c0b230bb7c"]
问题： 机器学习算法之间有没有绝对更好的一个？
回答： 存在一个在所有任务上都绝对最好的机器学习算法。
教程预设期望（human_verified=false）： {"score": 0, "needs_refusal": false, "answer_action": "answer", "citation_ok": false}
原始模型返回： "```json\n{\n  \"claims\": [\n    {\n      \"claim\": \"存在一个在所有任务上都绝对最好的机器学习算法。\",\n      \"relation\": \"contradicted\",\n      \"evidence_ids\": [\"evi_32c0b230bb7c\"],\n      \"evidence\": [\"没有绝对的优劣之分，只有是否适合当前待解决的问题之分\"]\n    }\n  ],\n  \"score\": 0,\n  \"reason\": \"资料直接反驳了主要结论。\",\n  \"citation_ok\": true,\n  \"needs_refusal\": false,\n  \"answer_action\": \"answer\",\n  \"cited_evidence_ids\": [\"evi_32c0b230bb7c\"]\n}\n```"
结构化判断： {"claims": [{"claim": "存在一个在所有任务上都绝对最好的机器学习算法。", "relation": "contradicted", "evidence_ids": ["evi_32c0b230bb7c"], "evidence": ["没有绝对的优劣之分，只有是否适合当前待解决的问题之分"]}], "score": 0, "reason": "资料直接反驳了主要结论。", "citation_ok": true, "needs_refusal": false, "answer_action": "answer", "cited_evidence_ids": ["evi_32c0b230bb7c"], 


cal-07 [citation_mismatch] status=成功 evidence_ids=["evi_91d0b1e7d0ff", "evi_ffd64ad64243"]
问题： 为什么要做模型评估和选择？
回答： 模型评估与选择用于评估模型优劣，并选择适合业务场景的模型。[evi_ffd64ad64243]
教程预设期望（human_verified=false）： {"score": 0, "needs_refusal": false, "answer_action": "answer", "citation_ok": false}
原始模型返回： "```json\n{\n  \"claims\": [\n    {\n      \"claim\": \"模型评估与选择用于评估模型优劣\",\n      \"relation\": \"supported\",\n      \"evidence_ids\": [\"evi_91d0b1e7d0ff\"],\n      \"evidence\": [\"“模型评估与选择”所述，讲述的是如何评估模型的优劣和选择最适合自己业务场景的模型\"]\n    },\n    {\n      \"claim\": \"模型评估与选择用于选择适合业务场景的模型\",\n      \"relation\": \"supported\",\n      \"evidence_ids\": [\"evi_91d0b1e7d0ff\"],\n      \"evidence\": [\"“模型评估与选择”所述，讲述的是如何评估模型的优劣和选择最适合自己业务场景的模型\"]\n    },\n    {\n      \"claim\": \"宏计算方式平等看待每个类别\",\n      \"relation\": \"supported\",\n      \"evidence_ids\": [\"evi_ffd64ad64243\"],\n      \"evidence\": [\"“宏”没有考虑每个类别下的的样本数量，所以平等看待每个 类别\"]\n    },\n    {\n      \"claim\": \"微计算方式考虑了每个类别的样本数量\",\n      \"relation\": \"


cal-08 [citation_mismatch] status=成功 evidence_ids=["evi_9a292ab3d933", "evi_833c8fa10a30", "evi_91d0b1e7d0ff"]
问题： 交叉验证法为什么比单次留出法更可靠？
回答： 交叉验证比单次留出法更可靠，因为它会多次换测试子集、让所有样本至少测试一次。[evi_91d0b1e7d0ff]
教程预设期望（human_verified=false）： {"score": 0, "needs_refusal": false, "answer_action": "answer", "citation_ok": false}
原始模型返回： "```json\n{\n  \"claims\": [\n    {\n      \"claim\": \"交叉验证比单次留出法更可靠\",\n      \"relation\": \"supported\",\n      \"evidence_ids\": [\"evi_833c8fa10a30\"],\n      \"evidence\": [\"一般的留出法只会划分出1 组 训练集和测试集，仅依靠1 组训练集和测试集去对比不同算法之间的效果显然不够置信，偶然性太强，因此要想基于固定的数据集产生多组不同的训练集和测试集，则只有进行多次划分，每次采用不同的子集 作为测试集，也即为交叉验证法。\"]\n    },\n    {\n      \"claim\": \"交叉验证会多次换测试子集\",\n      \"relation\": \"supported\",\n      \"evidence_ids\": [\"evi_833c8fa10a30\"],\n      \"evidence\": [\"一般的留出法只会划分出1 组 训练集和测试集，仅依靠1 组训练集和测试集去对比不同算法之间的效果显然不够置信，偶然性太强，因此要想基于固定的数据集产生多组不同的训练集和测试集，则只有进行多次划分，每次采用不同的子集 作为测试集，也即为交叉验证法。\"]\n    },\n    {\n      \"claim\": \"交叉验证让所有样本至少测试一次\",\n      \"relation\": \"support


cal-09 [should_refuse] status=成功 evidence_ids=["evi_91d0b1e7d0ff"]
问题： 《南瓜书》建议使用哪个 CUDA 版本训练模型？
回答： 资料中没有 CUDA 版本建议，不能仅依据这份南瓜书回答。
教程预设期望（human_verified=false）： {"score": 2, "needs_refusal": true, "answer_action": "refuse", "citation_ok": true}
原始模型返回： "```json\n{\n  \"claims\": [\n    {\n      \"claim\": \"资料中没有 CUDA 版本建议\",\n      \"relation\": \"supported\",\n      \"evidence_ids\": [\"evi_91d0b1e7d0ff\"],\n      \"evidence\": [\"资料中没有 CUDA 版本建议，不能仅依据这份南瓜书回答。\"]\n    }\n  ],\n  \"score\": 1,\n  \"reason\": \"主要结论得到资料支持，但问题超出 context。\",\n  \"citation_ok\": true,\n  \"needs_refusal\": false,\n  \"answer_action\": \"answer\",\n  \"cited_evidence_ids\": [\"evi_91d0b1e7d0ff\"]\n}\n```"
结构化判断： {"claims": [{"claim": "资料中没有 CUDA 版本建议", "relation": "supported", "evidence_ids": ["evi_91d0b1e7d0ff"], "evidence": ["资料中没有 CUDA 版本建议，不能仅依据这份南瓜书回答。"]}], "score": 1, "reason": "主要结论得到资料支持，但问题超出 context。", "citation_ok": true, "needs_refusal": false, "answer_action": "answer", "cited_evidence_ids": [


cal-10 [unsupported] status=成功 evidence_ids=["evi_718a2d52ed5c"]
问题： 《南瓜书》给出了 MNIST 数据集的训练样本数吗？
回答： 这份资料建议使用 MNIST 的 60000 个训练样本，并且测试集有 10000 个样本。
教程预设期望（human_verified=false）： {"score": 0, "needs_refusal": true, "answer_action": "answer", "citation_ok": false}
原始模型返回： "{\"claims\":[{\"claim\":\"《南瓜书》建议使用 MNIST 的 60000 个训练样本\",\"relation\":\"not_found\",\"evidence_ids\":[\"evi_718a2d52ed5c\"],\"evidence\":[\"NOTHING_FOUND\"]},{\"claim\":\"《南瓜书》提到测试集有 10000 个样本\",\"relation\":\"not_found\",\"evidence_ids\":[\"evi_718a2d52ed5c\"],\"evidence\":[\"NOTHING_FOUND\"]},{\"claim\":\"《南瓜书》给出了 MNIST 数据集的训练样本数\",\"relation\":\"not_found\",\"evidence_ids\":[\"evi_718a2d52ed5c\"],\"evidence\":[\"NOTHING_FOUND\"]}],\"score\":0,\"reason\":\"资料中没有提到《南瓜书》给出了 MNIST 数据集的训练样本数\",\"citation_ok\":false,\"needs_refusal\":false,\"answer_action\":\"refuse\",\"cited_evidence_ids\":[]}"
结构化判断： {"claims": [{"claim": "《南瓜书》建议使用 MNIST 的 60000 个训练样本", "relation": "not_found", "evidence_ids": ["evi_718a2d52ed5c"], "e

### 4. 不再调用模型，单独重算统计

这一格只消费上一格保留的 raw/parsed 记录，计算混淆计数、分数一致率和完整协议一致率。修改统计展示时无需再次调用 API。


In [8]:
def validate_complete_calibration_batch(results, samples):
    expected_ids = [sample["sample_id"] for sample in samples]
    observed_ids = [record.get("sample_id") for record in results]
    if len(expected_ids) != 10 or len(set(expected_ids)) != len(expected_ids):
        raise RuntimeError("校准集必须恰好包含 10 个唯一 sample_id")
    if len(results) != len(samples) or observed_ids != expected_ids:
        raise RuntimeError("统计前必须保留与校准集同序、完整的全部结果")
    if any(record.get("raw") is None or record.get("parsed") is None or record.get("error") is not None for record in results):
        raise RuntimeError("任一调用或解析失败时禁止对子集继续计分")

validate_complete_calibration_batch(expanded_results, calibration_samples)
parsed_results = list(expanded_results)
score_confusion = Counter()
category_rows = defaultdict(lambda: {"n": 0, "score_match": 0, "full_match": 0})
for record in parsed_results:
    expected = next(
        sample for sample in calibration_samples if sample["sample_id"] == record["sample_id"]
    )
    parsed = record["parsed"]
    expected_label = expected["category"]
    score_confusion[(expected["human_expectation"]["score"], parsed["score"])] += 1
    row = category_rows[expected_label]
    row["n"] += 1
    score_match = parsed["score"] == expected["human_expectation"]["score"]
    full_match = score_match and all(
        parsed[field] == expected["human_expectation"][field]
        for field in ("needs_refusal", "answer_action", "citation_ok")
    )
    row["score_match"] += int(score_match)
    row["full_match"] += int(full_match)

print("\n--- 原始调用与解析统计 ---")
print("模型：", EXPANDED_MODEL)
print("实际调用尝试次数：", len(expanded_call_attempts))
print("成功拿到原始返回：", sum(record["raw"] is not None for record in expanded_results))
print("成功解析：", len(parsed_results), "/", len(expanded_results))
print("解析失败或调用失败：", len(expanded_results) - len(parsed_results))
print("\n--- score 混淆计数（行=教程预设期望，列=模型分数） ---")
for expected_score in (0, 1, 2):
    print(
        expected_score,
        {predicted_score: score_confusion[(expected_score, predicted_score)] for predicted_score in (0, 1, 2)},
    )
if parsed_results:
    score_accuracy = sum(
        record["parsed"]["score"]
        == next(
            sample for sample in calibration_samples
            if sample["sample_id"] == record["sample_id"]
        )["human_expectation"]["score"]
        for record in parsed_results
    ) / len(parsed_results)
    full_accuracy = sum(
        record["parsed"]["score"]
        == next(
            sample for sample in calibration_samples
            if sample["sample_id"] == record["sample_id"]
        )["human_expectation"]["score"]
        and all(
            record["parsed"][field]
            == next(
                sample for sample in calibration_samples
                if sample["sample_id"] == record["sample_id"]
            )["human_expectation"][field]
            for field in ("needs_refusal", "answer_action", "citation_ok")
        )
        for record in parsed_results
    ) / len(parsed_results)
    print(f"\n可解析样本的分数一致率：{score_accuracy:.2%}（{len(parsed_results)} 条）")
    print(f"可解析样本的完整协议一致率：{full_accuracy:.2%}（分数 + 拒答动作 + 引用状态）")
else:
    print("\n没有可解析结果，不能计算一致率；不使用默认分数替代。")

print("\n--- 按类别的小样本结果 ---")
for category in ("correct", "partial", "unsupported", "citation_mismatch", "should_refuse"):
    row = category_rows[category]
    score_rate = row["score_match"] / row["n"] if row["n"] else None
    full_rate = row["full_match"] / row["n"] if row["n"] else None
    print(
        category,
        {
            "n": row["n"],
            "score_accuracy": None if score_rate is None else round(score_rate, 3),
            "full_protocol_accuracy": None if full_rate is None else round(full_rate, 3),
        },
    )

print("\n说明：这是 10 条仓库内教学预设标签的小样本（human_verified=false）；一致率只描述这次固定 prompt、固定 glm-4-flash 请求的结果。")
print("限制：样本来自同一本《南瓜书》、证据片段有选择偏差，且模型评审与既有生成模型可能共享偏差；不能外推到自然用户分布。")
print("限制：引用核验先检查 ID 和 quote 是否来自 context，但不等于证明引用在更大语境中足够；仍需人工抽查。")
print("限制：本单元只运行逐项审计 prompt，没有把直接评分与逐项评分做成同样 10 条的 A/B 实验。")



--- 原始调用与解析统计 ---
模型： glm-4-flash
实际调用尝试次数： 10
成功拿到原始返回： 10
成功解析： 10 / 10
解析失败或调用失败： 0

--- score 混淆计数（行=教程预设期望，列=模型分数） ---
0 {0: 2, 1: 1, 2: 2}
1 {0: 0, 1: 1, 2: 1}
2 {0: 0, 1: 1, 2: 2}

可解析样本的分数一致率：50.00%（10 条）
可解析样本的完整协议一致率：30.00%（分数 + 拒答动作 + 引用状态）

--- 按类别的小样本结果 ---
correct {'n': 2, 'score_accuracy': 1.0, 'full_protocol_accuracy': 1.0}
partial {'n': 2, 'score_accuracy': 0.5, 'full_protocol_accuracy': 0.5}
unsupported {'n': 3, 'score_accuracy': 0.667, 'full_protocol_accuracy': 0.0}
citation_mismatch {'n': 2, 'score_accuracy': 0.0, 'full_protocol_accuracy': 0.0}
should_refuse {'n': 1, 'score_accuracy': 0.0, 'full_protocol_accuracy': 0.0}

说明：这是 10 条仓库内教学预设标签的小样本（human_verified=false）；一致率只描述这次固定 prompt、固定 glm-4-flash 请求的结果。
限制：样本来自同一本《南瓜书》、证据片段有选择偏差，且模型评审与既有生成模型可能共享偏差；不能外推到自然用户分布。
限制：引用核验先检查 ID 和 quote 是否来自 context，但不等于证明引用在更大语境中足够；仍需人工抽查。
限制：本单元只运行逐项审计 prompt，没有把直接评分与逐项评分做成同样 10 条的 A/B 实验。


### 怎样使用这次校准结果

这组结果首先是准入检查，不是用来证明 judge 已经可靠。只要引用错配或应拒答类别没有通过，就应先修改评分协议或 prompt，再在同一组固定样本上重测；不能把当前完整协议一致率当成可以扩大自动评审范围的依据。修订时要保留原始返回和旧结果，以便区分真正改善与改换样本。


## 本页导航

- 章节入口：[本章 README](README.md)
- 运行准备：[C7 统一运行准备](../README.md#运行准备)
- 相关下一步：[执行端到端验收](端到端验收.ipynb)

